In [3]:
!pip install pytorchvideo torchvision av

In [4]:
# --- 0. THE PYTORCHVIDEO BUG FIX (Monkey Patch) ---
import sys
import torchvision.transforms.functional as F_vision
# Trick pytorchvideo into thinking the old deprecated module still exists
sys.modules['torchvision.transforms.functional_tensor'] = F_vision

# --- 1. IMPORTS ---
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from google.colab import drive

import pytorchvideo.data
from pytorchvideo.transforms import ApplyTransformToKey, UniformTemporalSubsample
from torchvision.transforms import Compose, Lambda, Resize
from torchvision.transforms._transforms_video import NormalizeVideo
from pytorchvideo.data.encoded_video import EncodedVideo

# --- 2. CONFIGURATION ---
drive.mount('/content/drive', force_remount=True)

DATA_PATH = "/content/drive/MyDrive/video_data/train"
CLASSES = ['walking', 'running']
BATCH_SIZE = 2
NUM_FRAMES = 16  # x3d_m defaults to 16 frames
EPOCHS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 3. ROBUST DATASET DEFINITION ---
labeled_video_paths = []
for i, cls_name in enumerate(CLASSES):
    cls_path = os.path.join(DATA_PATH, cls_name)
    if not os.path.isdir(cls_path): continue
    for f in os.listdir(cls_path):
        if f.lower().endswith(('.mp4', '.avi', '.mov')):
            labeled_video_paths.append((os.path.join(cls_path, f), {'label': i}))

# X3D Transform Pipeline
video_transform = Compose([
    ApplyTransformToKey(
        key="video",
        transform=Compose([
            UniformTemporalSubsample(NUM_FRAMES),
            Lambda(lambda x: x / 255.0), # Normalize to 0-1
            NormalizeVideo(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225]), # X3D specifics
            Resize((256, 256)) # X3D default crop size
        ])
    )
])

# Safely extract 2-second clips
clip_sampler = pytorchvideo.data.make_clip_sampler("uniform", 2.0)

train_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=labeled_video_paths,
    clip_sampler=clip_sampler,
    transform=video_transform,
    decode_audio=False
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE)

# --- 4. MODEL INITIALIZATION ---
print("Downloading X3D-M model from PyTorch Hub...")
model = torch.hub.load('facebookresearch/pytorchvideo', 'x3d_m', pretrained=True)

# Replace the classification head for X3D
in_features = model.blocks[5].proj.in_features
model.blocks[5].proj = nn.Linear(in_features, len(CLASSES))

model = model.to(DEVICE)

# --- 5. TRAINING LOOP ---
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

print("\n--- Starting X3D Training ---")
model.train()

for epoch in range(EPOCHS):
    running_loss = 0.0
    batches = 0

    for batch_dict in train_loader:
        # X3D expects shape: (Batch, Channels, Time, Height, Width)
        videos = batch_dict['video'].to(DEVICE)
        labels = batch_dict['label'].to(DEVICE)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        batches += 1

    avg_loss = running_loss / batches if batches > 0 else 0
    print(f"Epoch [{epoch+1}/{EPOCHS}] Average Loss: {avg_loss:.4f}")

# --- 5.1 SAVING THE TRAINED MODEL ---
save_path = "/content/drive/MyDrive/video_x3d_TrainedModel.pth"
torch.save(model.state_dict(), save_path)
print(f"Yay! Model safely saved to: {save_path}")

# --- 6. TARGET FILE / INFERENCE WITH THRESHOLD ---
VIDEO_PATH = "/content/drive/MyDrive/video_data/test/v_PlayingCello_g04_c02.avi"

def run_guaranteed_inference(video_path, threshold=0.95):
    """
    Runs inference and intercepts low-confidence predictions
    to label them as 'MISCELLANEOUS'.
    """
    model.eval()
    try:
        video = EncodedVideo.from_path(video_path)
        video_data = video.get_clip(start_sec=0.0, end_sec=2.0)

        # Apply the exact same transform
        video_data = video_transform(video_data)

        # Add batch dimension and send to GPU
        input_tensor = video_data["video"].unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1)
            conf, pred_idx = torch.max(probs, 1)

        # Extract the values
        confidence_score = conf.item()
        predicted_class = CLASSES[pred_idx.item()].upper()

        print("-" * 30)
        print(f"FILE: {os.path.basename(video_path)}")

        # --- THE MISCELLANEOUS INTERCEPT ---
        if confidence_score < threshold:
            print(f"PREDICTED: MISCELLANEOUS (Confidence below {threshold*100}%)")
            print(f"ORIGINAL GUESS: {predicted_class} at {confidence_score*100:.2f}%")
        else:
            print(f"PREDICTED: {predicted_class}")
            print(f"CONFIDENCE: {confidence_score*100:.2f}%")

        print("-" * 30)

    except Exception as e:
        print(f"Inference failed: {e}")

# Run the inference
run_guaranteed_inference(VIDEO_PATH, threshold=0.70)

Mounted at /content/drive


Using cache found in /root/.cache/torch/hub/facebookresearch_pytorchvideo_main



--- Starting X3D Training ---
Epoch [1/3] Average Loss: 0.6158
Epoch [2/3] Average Loss: 0.3998
Epoch [3/3] Average Loss: 0.3336
Yay! Model safely saved to: /content/drive/MyDrive/video_x3d_TrainedModel.pth
------------------------------
FILE: v_PlayingCello_g04_c02.avi
PREDICTED: RUNNING
CONFIDENCE: 71.56%
------------------------------
